In [1]:
!pip install elasticsearch==8.19.1 kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.5/940.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [elasticsearch]0m [elasticsearch]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:

from elasticsearch import Elasticsearch
import os
import pandas as pd



In [3]:
!docker cp search-system-es01-1:/usr/share/elasticsearch/config/certs/ca/ca.crt ./ca.crt

Successfully copied 3.07kB to /workspaces/search-system/search-db/ca.crt


In [4]:
!docker info | grep -i memory
!docker info | grep -i cpu

 Total Memory: 7.758GiB
 CPUs: 2


In [18]:

!free -h

               total        used        free      shared  buff/cache   available
Mem:           7.8Gi       5.8Gi       147Mi        64Mi       2.4Gi       2.0Gi
Swap:             0B          0B          0B


In [17]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay          32G   17G   13G  57% /
tmpfs            64M     0   64M   0% /dev
shm              64M  4.0K   64M   1% /dev/shm
/dev/root        29G   21G  8.0G  73% /vscode
/dev/sdc1        44G  4.2G   38G  10% /tmp
/dev/loop4       32G   17G   13G  57% /workspaces


In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thangndk67/sample-input-search-system")

print("Path to dataset files:", path)

100%|██████████| 2.04M/2.04M [00:01<00:00, 1.85MB/s]

Extracting files...


Path to dataset files: /home/codespace/.cache/kagglehub/datasets/thangndk67/sample-input-search-system/versions/1


In [9]:
client = Elasticsearch(
    hosts=["https://localhost:9200"],  # Địa chỉ Elasticsearch
    basic_auth=("elastic", "elastic"),
    request_timeout=60,
    ca_certs="./ca.crt"
)
client.info()

ObjectApiResponse({'name': 'es01', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'i6mIH4nrRriV0XuGZY73lQ', 'version': {'number': '8.19.4', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'aa0a7826e719b392e7782716b323c4fb8fa3b392', 'build_date': '2025-09-16T22:06:03.940754111Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [10]:

def get_all_file_names(folder_path):
    try:
        # List all files in the folder and remove ".json" extension
        file_names = [
            os.path.splitext(file)[0] for file in os.listdir(folder_path)
            if os.path.isfile(os.path.join(folder_path, file)) and (file.endswith(".ndjson") or file.endswith(".csv"))
        ]
        return file_names
    except FileNotFoundError:
        print(f"The folder '{folder_path}' does not exist.")
        return []

folder_path = path
file_names = get_all_file_names(folder_path)
print("Files in folder:", file_names)

Files in folder: ['output']


In [11]:
# template = {
#     "index_patterns": [
#         "wikipedia-people*"
#     ],
#     "template": {
#         "settings": {
#             "number_of_shards": 1,
#             "number_of_replicas": 0,
#             "refresh_interval": "60s",
#             "translog.durability": "async",
#             "translog.sync_interval": "30s",
#             "merge.scheduler.max_thread_count": 1,
#             "indexing.slowlog.threshold.index.warn": "10s",
#             "indexing.slowlog.threshold.index.info": "5s",
#             # "indexing.slowlog.level": "info"
#         },
#         "mappings": {
#           "properties": {
#             "identifier": { "type": "keyword" },
#             "name": {
#               "type": "text",
#               "fields": { "keyword": { "type": "keyword" } }
#             },
#             "description": { "type": "text" },
#             "abstract": { "type": "text" },
#             "full_text": { "type": "text" },
#             "image": {
#               "properties": {
#                 "content_url": { "type": "keyword" },
#                 "width": { "type": "integer" },
#                 "height": { "type": "integer" }
#               }
#             },
#             "categories": { "type": "keyword" },
#             "infobox": {
#               "type": "nested",
#               "properties": {
#                 "name": { "type": "keyword" },
#                 "value": { "type": "text" },
#                 "type": { "type": "keyword" }
#               }
#             },
#             "date_modified": { "type": "date" },
#             "url": { "type": "keyword" },
#             "main_entity": {
#               "properties": {
#                 "identifier": { "type": "keyword" },
#                 "url": { "type": "keyword" }
#               }
#             }
#           }
#       }
#     }
# }
#
#


In [12]:
# Tạo index template
import json
with open('index_template.json', 'r') as f:
    template = json.load(f)
client.indices.put_index_template(
    name="wikipedia_template",
    index_patterns=template['index_patterns'],
    template=template['template']
)

ObjectApiResponse({'acknowledged': True})

In [13]:
def extract_text_from_sections(sections):
    text_parts = []

    def extract_from_part(part):
        if isinstance(part, dict):
            # Xử lý phần có 'has_parts'
            if "has_parts" in part:
                for subpart in part["has_parts"]:
                    extract_from_part(subpart)

            # Xử lý phần có 'value' là chuỗi
            if "value" in part and isinstance(part["value"], str):
                cleaned_text = " ".join(part["value"].split())
                if cleaned_text:
                    text_parts.append(cleaned_text)

            # Xử lý list items
            if part.get("type") == "list_item" and "value" in part and isinstance(part["value"], str):
                cleaned_text = " ".join(part["value"].split())
                if cleaned_text:
                    text_parts.append(cleaned_text)

        elif isinstance(part, list):
            for item in part:
                extract_from_part(item)

    # Bắt đầu xử lý
    extract_from_part(sections)

    return " ".join(text_parts).strip()

In [16]:
from elasticsearch import helpers

# Process NDJSON files

# Create index (matches wikipedia-people* pattern)

INDEX_NAME = "wikipedia-people-sample"
if not client.indices.exists(index=INDEX_NAME):
    client.indices.create(index=INDEX_NAME)
    print(f"Index '{INDEX_NAME}' created successfully")

ndjson_dir = "./ndjson_files"  # Update to your NDJSON files directory
batch_size = 1000
actions = []

# for file_name in os.listdir(ndjson_dir):
#     if file_name.endswith(".ndjson"):
#         file_path = os.path.join(ndjson_dir, file_name)
with open(f"{path}/output.ndjson", "r", encoding="utf-8") as f:
    for line in f:

        try:
            doc = json.loads(line.strip())
        except json.JSONDecodeError:
            # print(f"Skipping malformed JSON in {file_name}")
            continue
        # print(doc)
        # break

        # Build full_text from sections
        full_text = extract_text_from_sections(doc.get("article_sections", []))
        # print(full_text)
        # break

        # Prepare ES document
        es_doc = {
            "_index": INDEX_NAME,
            "_id": str(doc.get("identifier")),  # Use identifier as doc ID
            "_source": {
                "identifier": doc.get("identifier"),
                "name": doc.get("name"),
                "description": doc.get("description"),
                "abstract": doc.get("abstract"),
                "full_text": full_text,
                "image": doc.get("image"),
                "categories": [cat["name"] for cat in doc.get("categories", [])],
                "infobox": doc.get("infobox", []),
                "date_modified": doc.get("date_modified"),
                "url": doc.get("url"),
                "main_entity": doc.get("main_entity")
            }
        }
        actions.append(es_doc)

        # Bulk index when batch is full
        if len(actions) >= batch_size:
            try:
                helpers.bulk(client, actions)
                # print(f"Indexed {len(actions)} documents from {file_name}")
                actions = []
            except Exception as e:
                print(f"Error indexing batch: {e}")

# Index any remaining documents
if actions:
    try:
        helpers.bulk(client, actions)
        print(f"Indexed final {len(actions)} documents")
    except Exception as e:
        print(f"Error indexing final batch: {e}")

print("Ingestion complete!")

Ingestion complete!
